# 07 — RAG Context Builder
Build a full RAG prompt using ContextBuilder, show before/after context injection, generate a response with ClaudeClient, and print source courses used.

In [ ]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = Path("../..")
sys.path.insert(0, str(REPO_ROOT / "src"))
load_dotenv(REPO_ROOT / ".env")

INDEX_DIR = REPO_ROOT / "data" / "domain"

from retrieval.retriever import CourseRetriever
from retrieval.context_builder import ContextBuilder
from llm.claude_client import ClaudeClient
from llm.cache import ResponseCache

retriever = CourseRetriever(index_dir=str(INDEX_DIR))
builder   = ContextBuilder(retriever, top_k=5)
cache     = ResponseCache(db_path=str(REPO_ROOT / "data" / "cache" / "responses.db"))
client    = ClaudeClient(model="claude-haiku-4-5", api_key=os.getenv("ANTHROPIC_API_KEY"))

print("All components loaded.")

## Prompt Before Context Injection

In [ ]:
question = "What courses should I take to prepare for graduate school in computer science?"

base_prompt = "You are a WSU academic advisor. Answer the student's question helpfully."

print("=== PROMPT WITHOUT CONTEXT ===")
print(f"{base_prompt}\n\nQuestion: {question}")

## Prompt After Context Injection

In [ ]:
full_prompt, sources = builder.build(question, base_prompt=base_prompt)

print("=== PROMPT WITH CONTEXT ===")
print(full_prompt)

## Generate Response with ClaudeClient
> Requires `ANTHROPIC_API_KEY` in `.env`. Uses cache so repeated runs are free.

In [ ]:
# Check cache first
cached = cache.get(full_prompt, model=client.model, temperature=0.0)
if cached:
    response = cached
    print("(served from cache)")
else:
    response = client.generate(full_prompt, temperature=0.0, max_tokens=600)
    cache.set(full_prompt, model=client.model, temperature=0.0, response=response)

print("\n=== RESPONSE ===")
print(response)

## Source Courses Used

In [ ]:
print(f"\n=== SOURCE COURSES ({len(sources)} retrieved) ===")
for i, s in enumerate(sources, 1):
    print(f"\n{i}. {s['course_code']} (similarity: {s['score']:.4f})")
    print(f"   {s['chunk_text'][:200]}")
    if s.get('prereq_raw'):
        print(f"   Prerequisites: {s['prereq_raw']}")

## Token Usage

In [ ]:
stats = client.get_usage_stats()
print("Token usage:", stats)